In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../Dashboards/powerbi_data/provider_dashboard_data.csv")

print(df.shape)
df.head()

(1296739, 16)


,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_first_name,rndrng_prvdr_type,rndrng_prvdr_state_abrvtn,tot_benes,tot_srvcs,tot_sbmtd_chrg,tot_mdcr_pymt_amt,payment_per_beneficiary,services_per_beneficiary,charge_to_payment_ratio,payment_vs_state_avg,payment_vs_specialty_avg,fraud_risk_score,risk_category
0,1003000126,Enkeshafi,Ardalan,Internal Medicine,MD,328,399.0,202783.88,35325.46,107.699573,1.216463,5.740446,0.277118,0.417696,3.852433,Low Risk
1,1003000134,Cibull,Thomas,Pathology,IL,912,2023.0,302581.00,50178.11,55.019857,2.218202,6.030139,0.528989,0.556196,8.821533,Low Risk
2,1003000142,Khalil,Rashid,Anesthesiology,OH,316,1369.0,381301.00,101577.94,321.449177,4.332278,3.753778,1.680158,3.070757,11.436628,Low Risk
3,1003000423,Velotta,Jennifer,Obstetrics & Gynecology,OH,75,140.0,22015.00,7134.32,95.124267,1.866667,3.085788,0.118006,0.400652,1.517909,Low Risk
4,1003000480,Rothchild,Kevin,General Surgery,CO,101,136.0,120420.00,18766.80,185.809901,1.346535,6.416651,0.242330,0.252014,2.780082,Low Risk


In [3]:
for i, col in enumerate(df.columns, start=1):
    print(i, col)

1 rndrng_npi
2 rndrng_prvdr_last_org_name
3 rndrng_prvdr_first_name
4 rndrng_prvdr_type
5 rndrng_prvdr_state_abrvtn
6 tot_benes
7 tot_srvcs
8 tot_sbmtd_chrg
9 tot_mdcr_pymt_amt
10 payment_per_beneficiary
11 services_per_beneficiary
12 charge_to_payment_ratio
13 payment_vs_state_avg
14 payment_vs_specialty_avg
15 fraud_risk_score
16 risk_category


In [4]:
print("Total Providers:", df["rndrng_npi"].nunique())
print("Total States:", df["rndrng_prvdr_state_abrvtn"].nunique())
print("Total Specialties:", df["rndrng_prvdr_type"].nunique())

print("\nRisk Category Distribution:")
print(df["risk_category"].value_counts())

Total Providers: 1296739
Total States: 62
Total Specialties: 113

Risk Category Distribution:
risk_category
Low Risk       1268774
Medium Risk      22345
High Risk         5620
Name: count, dtype: int64


In [5]:
def summarize_provider(npi):
    provider = df[df["rndrng_npi"] == npi]

    if provider.empty:
        return "No provider found with that NPI."

    row = provider.iloc[0]

    summary = f"""
Provider Investigation Summary

NPI: {row['rndrng_npi']}
Provider Name: {row['rndrng_prvdr_first_name']} {row['rndrng_prvdr_last_org_name']}
Specialty: {row['rndrng_prvdr_type']}
State: {row['rndrng_prvdr_state_abrvtn']}
Risk Category: {row['risk_category']}
Fraud Risk Score: {row['fraud_risk_score']:.2f}

Billing Metrics:
- Medicare Payments: ${row['tot_mdcr_pymt_amt']:,.2f}
- Total Beneficiaries: {row['tot_benes']:,.0f}
- Total Services: {row['tot_srvcs']:,.0f}
- Payment per Beneficiary: ${row['payment_per_beneficiary']:,.2f}
- Services per Beneficiary: {row['services_per_beneficiary']:.2f}
- Charge-to-Payment Ratio: {row['charge_to_payment_ratio']:.2f}
"""
    return summary

In [6]:
sample_npi = df["rndrng_npi"].iloc[0]

print(summarize_provider(sample_npi))


Provider Investigation Summary

NPI: 1003000126
Provider Name: Ardalan Enkeshafi
Specialty: Internal Medicine
State: MD
Risk Category: Low Risk
Fraud Risk Score: 3.85

Billing Metrics:
- Medicare Payments: $35,325.46
- Total Beneficiaries: 328
- Total Services: 399
- Payment per Beneficiary: $107.70
- Services per Beneficiary: 1.22
- Charge-to-Payment Ratio: 5.74



In [7]:
def get_high_risk_providers(limit=10):

    high_risk = (
        df[df["risk_category"] == "High Risk"]
        .sort_values("fraud_risk_score", ascending=False)
        .head(limit)
    )

    return high_risk[
        [
            "rndrng_npi",
            "rndrng_prvdr_last_org_name",
            "rndrng_prvdr_type",
            "rndrng_prvdr_state_abrvtn",
            "fraud_risk_score",
        ]
    ]

In [8]:
def providers_by_state(state):

    state = state.upper()

    result = (
        df[df["rndrng_prvdr_state_abrvtn"] == state]
        .sort_values("fraud_risk_score", ascending=False)
    )

    return result[
        [
            "rndrng_npi",
            "rndrng_prvdr_last_org_name",
            "rndrng_prvdr_type",
            "risk_category",
            "fraud_risk_score",
        ]
    ]

In [9]:
def providers_by_state(state):

    state = state.upper()

    result = (
        df[df["rndrng_prvdr_state_abrvtn"] == state]
        .sort_values("fraud_risk_score", ascending=False)
    )

    return result[
        [
            "rndrng_npi",
            "rndrng_prvdr_last_org_name",
            "rndrng_prvdr_type",
            "risk_category",
            "fraud_risk_score",
        ]
    ]

In [10]:
providers_by_state("CA")

,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_type,risk_category,fraud_risk_score
499709,1386627966,Orthopaedic Hospital,Pharmacy,High Risk,99.390273
464256,1356746267,Red Chip Of Nevada,Pharmacy,High Risk,98.874647
32202,1023296449,Drg Pharmacy Llc,Pharmacy,High Risk,98.837941
26039,1013998921,"Caremark, L.L.C.",Pharmacy,High Risk,98.837941
882025,1679959845,Mailyan,Family Practice,High Risk,98.661182
...,...,...,...,...,...
150386,1114506201,Keilman,Hospitalist,Low Risk,0.300524
1219522,1942208608,Block,Internal Medicine,Low Risk,0.285812
179081,1134680390,Bader,Internal Medicine,Low Risk,0.274837
477854,1366829475,Watson,Internal Medicine,Low Risk,0.269753


In [11]:
providers_by_state("TX")

,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_type,risk_category,fraud_risk_score
1065816,1821075060,Burrows,Neurology,High Risk,98.569635
229202,1174710636,The University Of Texas Health Science Center ...,Pharmacy,High Risk,98.507437
178391,1134633266,Prewett,Nurse Practitioner,High Risk,97.958087
1268143,1972839983,Reinauer,Ophthalmology,High Risk,97.126222
945423,1720621071,Lertdilok,Nurse Practitioner,High Risk,96.890595
...,...,...,...,...,...
313996,1245268168,Zuckerman,Gastroenterology,Low Risk,0.120204
770789,1598324741,Cashon,Physician Assistant,Low Risk,0.101614
768545,1598172041,Brown,Nurse Practitioner,Low Risk,0.095991
106802,1083208698,Hoelscher,Nurse Practitioner,Low Risk,0.082200


In [12]:
providers_by_state("IA")

,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_type,risk_category,fraud_risk_score
1278032,1982665337,Buroker,Medical Oncology,High Risk,97.162883
285480,1215999800,Behrens,Medical Oncology,High Risk,97.028445
280452,1215536636,Cook,Nurse Practitioner,High Risk,92.372836
563698,1437360351,Alliman,Ophthalmology,High Risk,92.278777
1236588,1952423071,Hussain,Hematology-Oncology,High Risk,92.190296
...,...,...,...,...,...
756574,1588233829,Schiller,Internal Medicine,Low Risk,0.372699
885362,1689168882,Wood,Family Practice,Low Risk,0.344695
1259084,1972092187,Witt,Hospitalist,Low Risk,0.278876
326435,1255326427,Hilgerson,Internal Medicine,Low Risk,0.257887


In [13]:
def providers_by_specialty(specialty):

    result = (
        df[df["rndrng_prvdr_type"]
        .str.contains(specialty, case=False, na=False)]
        .sort_values("fraud_risk_score", ascending=False)
    )

    return result[
        [
            "rndrng_npi",
            "rndrng_prvdr_last_org_name",
            "rndrng_prvdr_state_abrvtn",
            "risk_category",
            "fraud_risk_score",
        ]
    ]

In [14]:
providers_by_specialty("Cardiology")

,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_state_abrvtn,risk_category,fraud_risk_score
457764,1356360887,Emrani,CA,High Risk,93.450665
299830,1235142761,Li,CA,High Risk,91.050994
373157,1285749705,Bilkoo,FL,High Risk,90.291489
516746,1396833810,Kamboj,NV,High Risk,89.717822
832000,1649230988,Khan,CA,High Risk,87.932800
...,...,...,...,...,...
898419,1699182899,Turk,MO,Low Risk,0.499531
524304,1407340847,Ketcham,MI,Low Risk,0.376180
868211,1669886164,Mohebali,UT,Low Risk,0.310567
1091440,1841253259,Tinker,OK,Low Risk,0.262334


In [15]:
providers_by_specialty("Internal Medicine")

,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_state_abrvtn,risk_category,fraud_risk_score
850154,1659532927,Hedrick,IN,High Risk,100.000000
1238979,1952581027,Hardesty,IN,High Risk,100.000000
907148,1699907600,Lewandowska,IN,High Risk,98.617531
578785,1447591714,Ganz,FL,High Risk,98.191722
188180,1144516345,Karno,NV,High Risk,97.738866
...,...,...,...,...,...
334429,1255775722,Suraj,VA,Low Risk,0.065844
871315,1679101828,Khan,NM,Low Risk,0.059882
309292,1235709429,Maheshwari,IL,Low Risk,0.051492
911113,1700198587,Draghici,OR,Low Risk,0.040855


In [16]:
def search_providers(state=None, specialty=None, risk_category=None, limit=10):
    result = df.copy()

    if state:
        result = result[
            result["rndrng_prvdr_state_abrvtn"].str.upper() == state.upper()
        ]

    if specialty:
        result = result[
            result["rndrng_prvdr_type"].str.contains(
                specialty, case=False, na=False
            )
        ]

    if risk_category:
        result = result[
            result["risk_category"].str.lower() == risk_category.lower()
        ]

    result = result.sort_values("fraud_risk_score", ascending=False)

    return result[
        [
            "rndrng_npi",
            "rndrng_prvdr_last_org_name",
            "rndrng_prvdr_first_name",
            "rndrng_prvdr_type",
            "rndrng_prvdr_state_abrvtn",
            "risk_category",
            "fraud_risk_score",
            "tot_mdcr_pymt_amt",
            "tot_benes",
            "tot_srvcs",
        ]
    ].head(limit)

In [17]:
search_providers(
    state="TX",
    specialty="Cardiology",
    risk_category="High Risk",
    limit=5
)

,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_first_name,rndrng_prvdr_type,rndrng_prvdr_state_abrvtn,risk_category,fraud_risk_score,tot_mdcr_pymt_amt,tot_benes,tot_srvcs


In [18]:
def dashboard_summary():
    total_providers = df["rndrng_npi"].nunique()
    total_payments = df["tot_mdcr_pymt_amt"].sum()
    avg_risk = df["fraud_risk_score"].mean()
    high_risk_count = (df["risk_category"] == "High Risk").sum()

    top_state = (
        df[df["risk_category"] == "High Risk"]
        ["rndrng_prvdr_state_abrvtn"]
        .value_counts()
        .idxmax()
    )

    top_specialty = (
        df.groupby("rndrng_prvdr_type")["fraud_risk_score"]
        .mean()
        .sort_values(ascending=False)
        .idxmax()
    )

    summary = f"""
Healthcare Fraud Dashboard Summary

Total Providers: {total_providers:,}
Total Medicare Payments: ${total_payments:,.2f}
Average Fraud Risk Score: {avg_risk:.2f}
High-Risk Providers: {high_risk_count:,}

Key Findings:
- {top_state} has the highest number of high-risk providers.
- {top_specialty} has the highest average fraud risk score.
- High-risk providers should be prioritized for investigation based on fraud risk score and billing behavior.
"""

    return summary

In [19]:
print(dashboard_summary())


Healthcare Fraud Dashboard Summary

Total Providers: 1,296,739
Total Medicare Payments: $120,057,897,699.85
Average Fraud Risk Score: 11.11
High-Risk Providers: 5,620

Key Findings:
- CA has the highest number of high-risk providers.
- Radiation Therapy Center has the highest average fraud risk score.
- High-risk providers should be prioritized for investigation based on fraud risk score and billing behavior.



In [1]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1"

def ask_llama(prompt):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(OLLAMA_URL, json=payload)
    response.raise_for_status()

    return response.json()["response"]

In [2]:
print(ask_llama("Explain healthcare fraud in two sentences."))

Healthcare fraud is a type of white-collar crime where individuals or organizations intentionally deceive or mislead healthcare providers, payers (such as insurance companies), or government agencies to obtain money or benefits that are not owed to them. This can include billing for services that were never provided, upcoding or overcharging for procedures, or falsifying patient records to support false claims.


In [3]:
def get_data_context(question):
    q = question.lower()

    if "summary" in q or "dashboard" in q or "overview" in q:
        return dashboard_summary()

    elif "high risk" in q and "provider" in q:
        return get_high_risk_providers(limit=10).to_string(index=False)

    elif "california" in q or " ca " in q:
        return providers_by_state("CA").head(10).to_string(index=False)

    elif "texas" in q or " tx " in q:
        return providers_by_state("TX").head(10).to_string(index=False)

    elif "cardiology" in q:
        return providers_by_specialty("Cardiology").head(10).to_string(index=False)

    elif "internal medicine" in q:
        return providers_by_specialty("Internal Medicine").head(10).to_string(index=False)

    else:
        return dashboard_summary()

In [4]:
def ask_ai_assistant(question):
    data_context = get_data_context(question)

    prompt = f"""
You are an AI Healthcare Fraud Investigation Assistant.

Use only the provided healthcare fraud analytics data to answer the user's question.

User Question:
{question}

Relevant Data:
{data_context}

Instructions:
- Answer like a fraud analyst.
- Do not say fraud is proven.
- Use terms like "high-risk", "unusual billing pattern", and "recommended for review".
- Keep the answer clear and concise.
- If the data is tabular, summarize the most important providers or patterns.
"""

    return ask_llama(prompt)

In [7]:
%who function

ask_ai_assistant	 ask_llama	 dataframe_columns	 dataframe_hash	 dtypes_str	 get_data_context	 get_dataframes	 import_pandas_safely	 is_data_frame	 



In [10]:
import pandas as pd

df = pd.read_csv("../Dashboards/powerbi_data/provider_dashboard_data.csv")

In [11]:
print(df.shape)
df.head()

(1296739, 16)


,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_first_name,rndrng_prvdr_type,rndrng_prvdr_state_abrvtn,tot_benes,tot_srvcs,tot_sbmtd_chrg,tot_mdcr_pymt_amt,payment_per_beneficiary,services_per_beneficiary,charge_to_payment_ratio,payment_vs_state_avg,payment_vs_specialty_avg,fraud_risk_score,risk_category
0,1003000126,Enkeshafi,Ardalan,Internal Medicine,MD,328,399.0,202783.88,35325.46,107.699573,1.216463,5.740446,0.277118,0.417696,3.852433,Low Risk
1,1003000134,Cibull,Thomas,Pathology,IL,912,2023.0,302581.00,50178.11,55.019857,2.218202,6.030139,0.528989,0.556196,8.821533,Low Risk
2,1003000142,Khalil,Rashid,Anesthesiology,OH,316,1369.0,381301.00,101577.94,321.449177,4.332278,3.753778,1.680158,3.070757,11.436628,Low Risk
3,1003000423,Velotta,Jennifer,Obstetrics & Gynecology,OH,75,140.0,22015.00,7134.32,95.124267,1.866667,3.085788,0.118006,0.400652,1.517909,Low Risk
4,1003000480,Rothchild,Kevin,General Surgery,CO,101,136.0,120420.00,18766.80,185.809901,1.346535,6.416651,0.242330,0.252014,2.780082,Low Risk


In [12]:
print(dashboard_summary())


Healthcare Fraud Dashboard Summary

Total Providers: 1,296,739
Total Medicare Payments: $120,057,897,699.85
Average Fraud Risk Score: 11.11
High-Risk Providers: 5,620

Key Findings:
- CA has the highest number of high-risk providers.
- Radiation Therapy Center has the highest average fraud risk score.
- High-risk providers should be prioritized for review based on fraud risk score and billing behavior.



In [13]:
print(ask_ai_assistant("Summarize the dashboard"))

Based on the healthcare fraud analytics data, here's a summary of the dashboard:

**Overview**

Our analysis reveals 1.3 million providers with total Medicare payments exceeding $120 billion.

**High-Risk Providers**

5,620 providers are categorized as high-risk, with an average fraud risk score of 11.11. These providers should be prioritized for review due to their suspicious billing patterns.

**Regional Insights**

California has the highest number of high-risk providers, which warrants further investigation.

**Notable Providers**

Radiation Therapy Center stands out with the highest average fraud risk score, indicating an unusual billing pattern that may warrant review.

These findings highlight areas requiring closer examination to ensure Medicare payments are accurate and compliant. High-risk providers in California and Radiation Therapy Center should be prioritized for review.


In [14]:
print(ask_ai_assistant("Show high risk providers"))

NameError: name 'get_high_risk_providers' is not defined